# STAC data prep — Freshem Chloride (Zarr)

This notebook prepares the Freshem **chloride** Zarr datacube for publishing to the
Global Coastal Atlas STAC catalog. It follows the same pattern as
`notebooks/12_crop_production.ipynb`: read the already-processed data, inject
dataset metadata into the Zarr attrs, write a copy of the Zarr store next to a
`metadata_chloride.json` file, and verify the round trip.

**Prerequisite:** the Freshem pipeline (`freshem_stac/prepare_data.py`) must already
have produced `chloride.zarr` (see `C:\Ocean\Work\Projects\2026\Freshem\Tools\freshem_stac`).

Unlike the crop-production HIS datasets, this is a gridded `(x, y, z)` datacube in the
Dutch RD New projection (EPSG:28992), not a station time-series — there is no `time`
dimension and no vertical interpolation: `z` indexes the original, non-uniform model
layers (several `z` can share the same `top`/`bottom`).

Run `scripts/16_chloride.ipynb` afterwards to build the STAC collection/items.

In [7]:
import json
from pathlib import Path

import xarray as xr


## 1) Configure paths

Update these if the Freshem outputs move.

In [8]:
import os

freshem_repo = Path(os.environ.get("FRESHEM_REPO", r"C:\Ocean\Work\Projects\2026\Freshem\Tools\freshem_stac"))
stac_repo = Path(os.environ.get("STAC_DATA_REPO", r"C:\Ocean\Work\Projects\2025\STAC\Tools\Repositories\global-coastal-atlas\STAC\data"))

source_zarr = freshem_repo / "processed" / "chloride.zarr"
output_dir = freshem_repo / "processed" / "stac_folder" / "chloride"
output_zarr = output_dir / "chloride.zarr"

metadata_template_path = stac_repo / "metadata" / "metadata_chloride.json"
metadata_output_path = output_dir / "metadata_chloride.json"
varmap_output_path = output_dir / "chloride_var_name_map.json"

output_dir.mkdir(parents=True, exist_ok=True)

if not source_zarr.exists():
    raise FileNotFoundError(
        f"Source Zarr not found: {source_zarr}. Run freshem_stac/prepare_data.py first."
    )
if not metadata_template_path.is_file():
    raise FileNotFoundError(f"Metadata template not found: {metadata_template_path}")

print("Source Zarr:", source_zarr)
print("Output Zarr:", output_zarr)
print("Metadata template:", metadata_template_path)


Source Zarr: /workspace/freshem_stac/processed/chloride.zarr
Output Zarr: /workspace/freshem_stac/processed/stac_folder/chloride/chloride.zarr
Metadata template: /workspace/global-coastal-atlas/STAC/data/metadata/metadata_chloride.json


## 2) Zarr-safe variable name map

`prepare_data.py` already renamed variables that collide case-insensitively on
Windows/NTFS (e.g. `Z` -> `Z_orig`). We keep that rename (it must stay, the
store is written to disk again here) but record the original label in a sidecar
map and in `ds.attrs`, the same way `HIS_VAR_NAME_MAP` is used for the crop
production HIS variables, so the STAC item / frontend can show the original
name if desired.

In [9]:
ORIGINAL_NAME_MAP = {'z_orig': 'z'}  # safe_name -> original_name


def save_var_name_map(path: Path, name_map: dict) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(name_map, f, indent=2)


def restore_original_var_names(ds: xr.Dataset, safe_to_original: dict | None = None) -> xr.Dataset:
    """Rename Zarr-safe vars back to their original Freshem names."""
    if safe_to_original is None:
        raw = ds.attrs.get("FRESHEM_VAR_NAME_MAP")
        if not raw:
            return ds
        safe_to_original = json.loads(raw)
    rename = {
        k: v
        for k, v in safe_to_original.items()
        if k in ds.data_vars and v not in ds.dims and v not in ds.coords and v not in ds.data_vars
    }
    skipped = {
        k: v for k, v in safe_to_original.items() if k in ds.data_vars and k not in rename
    }
    if skipped:
        print(
            "Not restoring original name(s) because they collide with an existing "
            f"dimension/variable: {skipped}. Keeping the Zarr-safe name(s)."
        )
    return ds.rename(rename) if rename else ds


## 3) Load source data and refresh metadata

`SPATIAL_EXTENT` is recomputed from the actual grid (reprojected from the native
EPSG:28992 grid to EPSG:4326, as required for STAC bboxes), together with
`DIMENSIONS`, `SPATIAL_RESOLUTION`, `NODATA` and `DATA_TYPE`.

In [10]:
with open(metadata_template_path, encoding="utf-8") as f:
    metadata = json.load(f)

ds = xr.open_zarr(source_zarr, consolidated=False)

try:
    from pyproj import Transformer

    transformer = Transformer.from_crs(metadata["CRS"], "EPSG:4326", always_xy=True)
    x0, x1 = float(ds.x.min()), float(ds.x.max())
    y0, y1 = float(ds.y.min()), float(ds.y.max())
    lon0, lat0 = transformer.transform(x0, y0)
    lon1, lat1 = transformer.transform(x1, y1)
    metadata["SPATIAL_EXTENT"] = [
        round(min(lon0, lon1), 6),
        round(min(lat0, lat1), 6),
        round(max(lon0, lon1), 6),
        round(max(lat0, lat1), 6),
    ]
except ImportError:
    print("pyproj not available, keeping SPATIAL_EXTENT from the metadata template")

x_res = float(ds.x[1] - ds.x[0]) if ds.sizes["x"] > 1 else None
metadata["DIMENSIONS"] = ["x", "y", "z"]
metadata["SPATIAL_RESOLUTION"] = abs(x_res) if x_res is not None else metadata.get("SPATIAL_RESOLUTION")
metadata["DATA_TYPE"] = str(ds["cl_mean"].dtype)
metadata["NODATA"] = None

print("Refreshed SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])
print("Refreshed SPATIAL_RESOLUTION:", metadata["SPATIAL_RESOLUTION"])
print("Refreshed DATA_TYPE:", metadata["DATA_TYPE"])
print("Variables:", list(ds.data_vars))
print("Dims:", dict(ds.sizes))


Refreshed SPATIAL_EXTENT: [3.64028, 51.375148, 3.960967, 51.616396]
Refreshed SPATIAL_RESOLUTION: 50.0
Refreshed DATA_TYPE: float32
Variables: ['cl_q2', 'bottom', 'cl_q15', 'cl_q30', 'cl_q25', 'cl_q10', 'cl_q35', 'cl_q1', 'cl_q20', 'cl_mean', 'cl_q45', 'cl_q40', 'cl_q50', 'cl_q5', 'cl_q60', 'cl_q65', 'cl_q55', 'cl_q80', 'cl_q70', 'cl_q75', 'cl_q90', 'cl_q85', 'used_geotop', 'cl_q98', 'cl_q95', 'cl_q99', 'top', 'z_orig']
Dims: {'z': 70, 'y': 528, 'x': 458}


## 4) Inject metadata into Zarr attrs and write the output store

In [11]:
ds.attrs.update(metadata)
ds.attrs["Conventions"] = "CF-1.8"
ds.attrs["FRESHEM_VAR_NAME_MAP"] = json.dumps(ORIGINAL_NAME_MAP)

# Drop the encoding inherited from the source (Zarr v3) store: v3-only keys
# like "serializer" are rejected when writing Zarr v2 below.
for name in ds.variables:
    ds[name].encoding = {}

# zarr_format=2 keeps the output readable by the repo's "coclico" conda env
# (zarr-python 2.x), which cannot read Zarr v3 stores (zarr.json layout).
ds.to_zarr(output_zarr, mode="w", zarr_format=2)
save_var_name_map(varmap_output_path, ORIGINAL_NAME_MAP)

with open(metadata_output_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("Wrote Zarr store:", output_zarr)
print("Wrote metadata:", metadata_output_path)
print("Wrote var name map:", varmap_output_path)


Wrote Zarr store: /workspace/freshem_stac/processed/stac_folder/chloride/chloride.zarr
Wrote metadata: /workspace/freshem_stac/processed/stac_folder/chloride/metadata_chloride.json
Wrote var name map: /workspace/freshem_stac/processed/stac_folder/chloride/chloride_var_name_map.json


## 5) Verify the round trip

In [12]:
verify_ds = xr.open_zarr(output_zarr, consolidated=False)
assert set(verify_ds.data_vars) == set(ds.data_vars), "Variable names changed unexpectedly"
assert verify_ds.attrs.get("Conventions") == "CF-1.8"

restored = restore_original_var_names(verify_ds)
print("Stored variable names:", sorted(verify_ds.data_vars))
print("Restored (original) names:", sorted(restored.data_vars))
print("COLLECTION_ID:", verify_ds.attrs["COLLECTION_ID"])
print("OK: {dataset} Zarr store and metadata are ready for the STAC builder notebook.")


Not restoring original name(s) because they collide with an existing dimension/variable: {'z_orig': 'z'}. Keeping the Zarr-safe name(s).
Stored variable names: ['bottom', 'cl_mean', 'cl_q1', 'cl_q10', 'cl_q15', 'cl_q2', 'cl_q20', 'cl_q25', 'cl_q30', 'cl_q35', 'cl_q40', 'cl_q45', 'cl_q5', 'cl_q50', 'cl_q55', 'cl_q60', 'cl_q65', 'cl_q70', 'cl_q75', 'cl_q80', 'cl_q85', 'cl_q90', 'cl_q95', 'cl_q98', 'cl_q99', 'top', 'used_geotop', 'z_orig']
Restored (original) names: ['bottom', 'cl_mean', 'cl_q1', 'cl_q10', 'cl_q15', 'cl_q2', 'cl_q20', 'cl_q25', 'cl_q30', 'cl_q35', 'cl_q40', 'cl_q45', 'cl_q5', 'cl_q50', 'cl_q55', 'cl_q60', 'cl_q65', 'cl_q70', 'cl_q75', 'cl_q80', 'cl_q85', 'cl_q90', 'cl_q95', 'cl_q98', 'cl_q99', 'top', 'used_geotop', 'z_orig']
COLLECTION_ID: freshem_chloride
OK: {dataset} Zarr store and metadata are ready for the STAC builder notebook.
